# Layer 1: Pose Extraction

Extract 33-joint coordinates from the master video using MediaPipe Pose,
then visually verify extraction quality via skeleton overlay video.

**Input**: `sample_videos/chon_ji_master.mp4`  
**Output**:
- `sample_videos/chon_ji_master_poses.json` — per-frame joint coordinates
- `sample_videos/chon_ji_master_overlay.mp4` — skeleton overlay video

In [ ]:
import sys
import pathlib

# Add project root to sys.path (two levels up from notebooks/)
project_root = pathlib.Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root: {project_root}")

In [ ]:
from itf_analysis.pose_extraction.extractor import (
    extract_poses_from_video,
    save_poses_to_json,
    visualize_pose_overlay,
)

VIDEO_PATH   = str(project_root / "itf_analysis" / "sample_videos" / "chon_ji_master.mp4")
JSON_OUT     = str(project_root / "itf_analysis" / "sample_videos" / "chon_ji_master_poses.json")
OVERLAY_OUT  = str(project_root / "itf_analysis" / "sample_videos" / "chon_ji_master_overlay.mp4")

print("video path:", VIDEO_PATH)

In [ ]:
# Run pose extraction
frames = extract_poses_from_video(VIDEO_PATH)

total    = len(frames)
detected = sum(1 for f in frames if f.landmarks)
print(f"\ntotal frames : {total}")
print(f"pose detected: {detected} ({detected/total*100:.1f}%)")
print(f"no detection : {total - detected}")

In [ ]:
# Inspect a sample frame
first_detected = next(f for f in frames if f.landmarks)
print(f"=== frame_index={first_detected.frame_index}, t={first_detected.timestamp_ms:.0f}ms ===")
print(f"landmarks: {len(first_detected.landmarks)}")

key_joints = {
    11: 'left shoulder',  12: 'right shoulder',
    23: 'left hip',       24: 'right hip',
    25: 'left knee',      26: 'right knee',
    27: 'left ankle',     28: 'right ankle',
}
for idx, name in key_joints.items():
    lm = first_detected.landmarks.get(idx)
    if lm:
        print(f"  {idx:2d} {name}: x={lm.x:.3f} y={lm.y:.3f} z={lm.z:.3f} vis={lm.visibility:.2f}")

In [ ]:
# Save to JSON
save_poses_to_json(frames, JSON_OUT)

In [ ]:
# Generate overlay video
visualize_pose_overlay(VIDEO_PATH, frames, OVERLAY_OUT)

In [ ]:
# Display a mid-point frame from the overlay video inline
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(OVERLAY_OUT)

mid = len(frames) // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
ret, bgr = cap.read()
cap.release()

if ret:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(rgb)
    plt.title(f"Overlay frame #{mid}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Could not read overlay video.")

## Validation Checklist

- [ ] Pose detection rate > 80%
- [ ] `frames[0].landmarks` contains all 33 joints
- [ ] JSON file saved successfully
- [ ] Skeleton aligns accurately with the master's body in the overlay video
- [ ] Skeleton is maintained through fast transition frames

All items passing → proceed to Layer 2 (normalization).